# Gebäudesektormodell – Simulationsläufe mit Parquet analysieren

Dieses interne Notebook zeigt die Paketfunktionen für Jahresanalysen. Eine Zeile repräsentiert ein Szenario. Die gespeicherten Presets dienen zur Auswahl und Gruppierung.

Benötigte Umgebung einmalig im Repository installieren mit `python -m pip install -e ".[analysis]"`. Als Kernel dieselbe Python-Umgebung auswählen.

Der Parameter `TYPOLOGY_DIRS` enthält die Exportordner mit `batch.json` und `annual-*.xlsx`. Je Lauf wird ein Parquet erzeugt oder wiederverwendet. `REBUILD_PARQUET=True` wiederholt diese Konvertierungen. `CATALOG_DIR` enthält den gemeinsamen Katalog, der aus den angegebenen Laufdateien neu gebaut wird. Unterschiedliche Läufe behalten ihre eigene `batch`-Identität.

Die neuen öffentlichen Einstiegspunkte verwenden DataFrames und vorhandene Modellobjekte. Das Parquet speichert die bereits vorhandenen Jahreswerte kompakt, damit nur benötigte Spalten gelesen werden müssen. Es wird über die Paketfunktion erneuert. Die fachlichen Definitionen teilen sich Notebook und andere Anwendungen. Zusätzlicher Pflegebedarf entsteht bei Schemaänderungen und neuen Reports.

Der Katalog ist eine abgeleitete Kopie der Laufdateien und erleichtert das gemeinsame Laden und Weitergeben. `combine_annual_batches()` übernimmt Typabgleich, Herkunft und Schreiben in Blöcken. Nach neuen Läufen wird er aus der expliziten Quellenliste neu erzeugt. Damit bleibt diese wiederverwendbare Logik im Paket.

In [ ]:
from pathlib import Path
import logging
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display
from pexl import reporting, plot
from pexl.io.parquet import export_annual_batch, combine_annual_batches
from pexl.schema.current import CHARTS

ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").exists() and (path / "src/pexl").is_dir())
TYPOLOGY_DIRS = [
    ROOT / "data/exports/typologies/building_sector_v2",
    ROOT / "data/exports/typologies/building_sector_v2_keinePV",
]
CATALOG_DIR = ROOT / "data/building_typologies/building_sector_v2_incl_PV"
REBUILD_PARQUET = False
LANGUAGE = "de"
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("building_sector_analysis")

## Simulationslauf zusammenfassen

Die Paketfunktion liest die aufgezeichneten Chunks in zwei Durchläufen und schreibt begrenzte Gruppen nach Parquet. Bei 1.296 Excel-Dateien kann das mehrere Minuten dauern. Logging zeigt den Fortschritt. Die Ausgabe enthält `batch`, `scenario_name`, `source_file` und die originalen Variablen. Neue Exporte enthalten zusätzlich Herkunft und Schemastand in den Parquet-Metadaten.

Ein bereits vorhandenes Parquet wird hier bewusst wiederverwendet. Nach einer Änderung der Quelldaten `REBUILD_PARQUET` aktivieren. Teilweise abgeschlossene Läufe werden mit Warnung als Teilbestand exportiert.

Anschließend fasst die Bibliothek die Laufdateien zusammen. Numerische Datentypen werden vereinheitlicht, fehlende Spalten mit null ergänzt und geloggt. Der Index `(batch, scenario_name)` trennt identische Namen in verschiedenen Läufen. Doppelte vollständige Identitäten verhindern einen fehlerhaften Katalog. Bestehende Kataloge werden erst nach erfolgreichem Abschluss ersetzt.

In [ ]:
run_paths = []
for typology_dir in TYPOLOGY_DIRS:
    output_dir = ROOT / "data/building_typologies" / typology_dir.name
    run_path = output_dir / "scenarios.parquet"
    if REBUILD_PARQUET or not run_path.exists():
        run_path = export_annual_batch(typology_dir, output_dir)
    else:
        logger.info("Verwende vorhandenen Lauf: %s", run_path)
    run_paths.append(run_path)

parquet_path = combine_annual_batches(run_paths, CATALOG_DIR)
parquet = pq.ParquetFile(parquet_path)
display({"Szenarien": parquet.metadata.num_rows,
         "Spalten": parquet.metadata.num_columns,
         "Datei": str(parquet_path)})

## Benötigte Jahreswerte laden

Für das Showcase werden die Beiträge aller implementierten Jahresreports und die Filterspalten geladen. Normale Einzelanalysen können die Spaltenauswahl weiter reduzieren. GWP-Spalten enthalten auch Felder für die bestehende Legacy-Aufbereitung.

`pd.read_parquet(..., filters=[("PV_roof_share", "=", 0.3)])` kann bereits beim Lesen einen Teilbestand auswählen. Als stabile Zeilenidentität verwenden wir Lauf und Szenarioname.

In [ ]:
available = parquet.schema_arrow.names
preset_columns = [name for name in available if name.startswith("preset_recorded_")]
report_columns = {series.var_name for report in CHARTS.values() for series in report.series}
analysis_columns = {"batch", "scenario_name", "source_file", "tool_version", "FSI", "GFA_total",
                    "PV_roof_share", "QH", "QC", "PEI_balance", "PEI_virtual_demand",
                    "PEI_virtual_supply", "weather_name"}
gwp_columns = {name for name in available if name.startswith(("GWP_", "legacy_GWP_"))}
columns = [name for name in available
           if name in report_columns | analysis_columns | gwp_columns or name in preset_columns]
df = pd.read_parquet(parquet_path, columns=columns).set_index(["batch", "scenario_name"])
logger.info("%d Szenarien, %d geladene Spalten, %.1f MiB DataFrame",
            len(df), len(df.columns), df.memory_usage(deep=True).sum() / 1024**2)
display(df.head(3))

# Die Laufnamen allein ersetzen keinen Blick auf die tatsächlich gespeicherten PV-Anteile.
display(df.reset_index().groupby(["batch", "PV_roof_share"], dropna=False)
        .size().rename("Szenarien").to_frame())

## Preset-Kategorien erkunden und gemeinsam filtern

Die thermische Hülle steht im aktuellen Export unter `preset_recorded_hulls`, die Geometrie unter `preset_recorded_geometry`. Ein Presetname ist ein gespeicherter Wert und bleibt bei einem Sprachwechsel unverändert.

Das Beispiel wählt je zwei verfügbare Werte für Geometrie, Hülle und Heizsystem. Die Listen können durch konkrete Werte aus der angezeigten Übersicht ersetzt werden. `isin()` bedeutet ODER innerhalb einer Kategorie. Das gemeinsame `all(axis=1)` bedeutet UND zwischen Kategorien.

In [ ]:
categories = ["preset_recorded_geometry", "preset_recorded_hulls",
              "preset_recorded_heating_system"]
for category in categories:
    display(df[category].value_counts(dropna=False).rename(category))
filters = {category: df[category].dropna().unique()[:2].tolist() for category in categories}
mask = pd.DataFrame({column: df[column].isin(values)
                     for column, values in filters.items()}, index=df.index).all(axis=1)
selected = df.loc[mask].copy()
display(filters)
display({"Gesamt": len(df), "Auswahl": len(selected)})
display(selected[categories + ["PV_roof_share", "QH", "QC"]].head(12))

## Histogramm, Boxplot und Violinplot

Die Diagramme beschreiben ungewichtete simulierte Varianten. Die Verteilung entspricht keiner Gewichtung des Gebäudebestands. Die gleichen Plotfunktionen können direkt mit beliebigen DataFrame-Spalten arbeiten. `reporting.variables()` ergänzt die Schema-Beschriftungen und Einheiten.

In [ ]:
data = reporting.variables(df, "QH", language=LANGUAGE)
display(plot.histogram(**data, variable="QH", bins=40))
data = reporting.variables(selected, "QH", "preset_recorded_hulls", language=LANGUAGE)
display(plot.histogram(**data, variable="QH", color="preset_recorded_hulls", bins=30))
display(plot.distribution(**data, variable="QH", by="preset_recorded_hulls", kind="box"))
data = reporting.variables(selected, "QC", "preset_recorded_geometry", language=LANGUAGE)
display(plot.distribution(**data, variable="QC", by="preset_recorded_geometry", kind="violin"))

## XY-Diagramme und fachliche Shortcuts

`primary_energy_scatter_plot()` stellt die gespeicherten virtuellen Bedarf-/Deckungssummen gegenüber. `density_balance_scatter_plot()` verwendet GFZ und den gespeicherten Primärenergiesaldo. Diese Größen haben die Vorzeichenkonvention des Exports. Die später gezeigten Reports bilden ihre Beiträge gemäß ihrer jeweiligen Reportdefinition ab.

In [ ]:
display(reporting.primary_energy_scatter_plot(
    df, color="preset_recorded_heating_system", language=LANGUAGE))
display(reporting.primary_energy_scatter_plot(
    selected, color="preset_recorded_heating_system", symbol="preset_recorded_geometry",
    size="GFA_total", hover_data=("FSI", "PV_roof_share", "preset_recorded_hulls"),
    language=LANGUAGE))
display(reporting.density_balance_scatter_plot(
    selected, color="preset_recorded_heating_system", language=LANGUAGE))
data = reporting.variables(selected, "QH", "QC", "preset_recorded_hulls", language=LANGUAGE)
display(plot.scatter(**data, x="QH", y="QC", color="preset_recorded_hulls"))

## Kontrollierte Auswahl für Bilanzvergleiche

Für einen übersichtlichen Detailvergleich variieren wir nur das Hüllpreset innerhalb der oben gewählten Kategorien. Alle anderen gespeicherten Preset-Kategorien sowie PV-Anteil und Wettername werden auf den ersten Treffer festgelegt. Damit ist die Auswahl besser kontrolliert. Für eine kausale Interpretation sollten bei Bedarf weitere IN-Parameter berücksichtigt werden.

Die Vergleichsdiagramme zeigen maximal acht Treffer. Die vollständige Auswahl bleibt im DataFrame `comparison` verfügbar.

In [ ]:
if selected.empty:
    raise ValueError("Die gewählte Preset-Kombination hat keine Treffer. Filterlisten anpassen.")
reference = selected.iloc[0]
controls = [name for name in preset_columns if name != "preset_recorded_hulls"]
controls += [name for name in ("PV_roof_share", "weather_name") if name in selected]
control_match = selected[controls].eq(reference[controls]) | (
    selected[controls].isna() & reference[controls].isna())
comparison = selected.loc[control_match.all(axis=1)].copy()
comparison_plot = comparison.head(8)
display(comparison[categories + ["PV_roof_share", "weather_name", "QH", "QC"]])

## Wärme-, Endenergie- und Primärenergiebilanzen

Die Shortcuts kennen die benötigten Beiträge. Beide Sprachvarianten verwenden dieselben Werte und Serien. Daten können auch explizit mit `reporting.materialize()` aufbereitet und anschließend über `plot.reports.render()` dargestellt werden.

In [ ]:
display(reporting.heat_balance_plot(comparison_plot, language=LANGUAGE))
display(reporting.final_energy_balance_plot(comparison_plot, language=LANGUAGE))
display(reporting.primary_energy_balance_plot(comparison_plot, language=LANGUAGE, annotate=True))
display(reporting.heat_balance_plot(comparison_plot, language="en"))

## GWP-Berichte und Beiträge

Das Showcase umfasst die Lebenszyklus-Gegenüberstellung, die Detailbeiträge der grauen Emissionen, die fossil/biogene Aufteilung und den vierteiligen Quartiersbericht. Die vorhandenen GWP-Regeln bestimmen, welche Datenfamilie verfügbar ist. Für historische Bestände kann insbesondere die fossil/biogene Aufteilung fehlen.

In [ ]:
display(reporting.gwp_ee_plot(comparison_plot, language=LANGUAGE))
display(reporting.gwp_ee_detail_plot(comparison_plot, language=LANGUAGE))
display(reporting.gwp_ee_fossil_biogenic_plot(comparison_plot, language=LANGUAGE))
display(reporting.district_gwp_ee_plot(comparison_plot, language="en"))

## Reportsaldo als Datensatz für weitere Analysen

`reporting.balance()` liefert Bedarf, Deckung und Deckung minus Bedarf aus den Reportbeiträgen. Zusätzliche Spalten aus `include` ermöglichen beliebige Vergleiche. Die Werte bleiben mit ihrer Szenarioidentität verbunden. So lassen sich dieselben fachlichen Ergebnisse als Scatter oder Histogramm darstellen.

In [ ]:
data = reporting.balance(selected, "final_energy_balance",
                         include=("FSI", "preset_recorded_heating_system"), language=LANGUAGE)
display(data["frame"].head())
display(plot.scatter(**data, x="FSI", y="balance", color="preset_recorded_heating_system"))
display(plot.histogram(**data, variable="balance", bins=30))
report = reporting.materialize(comparison_plot, "heat_balance", language=LANGUAGE)
display(report["frame"])
from pexl.plot import reports as report_plots
display(report_plots.render(report))

## Dieselben Funktionen mit Project verwenden

Wenn bereits ein `Project` geladen ist, kann es an dieselben Shortcuts übergeben werden, beispielsweise `reporting.heat_balance_plot(project, language="de")`. Auch ausgewählte `ProjectView`-Objekte werden unterstützt. Die Analyse dieses Notebooks verwendet durchgehend den Parquet-DataFrame.

Öffentliche Funktionen dokumentieren ihre Parameter und Rückgaben ausführlich. In Jupyter zeigt `help(reporting.balance)` beziehungsweise `help(export_annual_batch)` die Details. Warnungen weisen unter anderem auf fehlende Spalten und abweichende Modellversionen hin. Die Funktionen ersetzen keine fachliche Validierung der Simulation.